#### 2026-01-30_clean_CIMMYT_farmer_plots.ipynb
- **Author:** Jay Sayre
- **Email:** jsayre@ucdavis.edu
- **Date modified:** 2026-01-30
- **Description:** Cleans the CIMMYT Farmer_plots data (logbook and sowing/harvest/yields files) and produces two outputs in Intermediates/CIMMYT:
    1. A shapefile of unique plot/farm locations (point geometries from GPS coordinates)
    2. A CSV of maize grain yields (tons/ha) by plot, year, and growing season (Spring-Summer vs. Fall-Winter)

- **Inputs:**
    - `data/CIMMYT/Farmer_plots/1.-Farmer_Plot_Logbook_2012-2022_03.xlsx` — Plot-level info including GPS coordinates, state, municipality
    - `data/CIMMYT/Farmer_plots/2.-Sowing_harvest_yields_2012-2022_02.xlsx` — Sowing, harvest, and yield records by plot-year-season

- **Outputs:**
    - `Intermediates/CIMMYT/cimmyt_plot_locations.shp` (+ .shx, .dbf, .prj, .cpg) — Shapefile of plot point locations
    - `Intermediates/CIMMYT/cimmyt_maize_yields.csv` — Maize grain yield (tons/ha) by plot, year, and season

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from   shapely.geometry import Point

# ── Directories ──────────────────────────────────────────────────────────────
top_dir      =  os.path.join(os.path.expanduser('~'), 'Dropbox', 'Projects')
proj_dir     =  os.path.join(top_dir, 'Maize_prediction')
cimmyt_dir   =  os.path.join(proj_dir, 'Data', 'CIMMYT', 'Farmer_plots')
out_dir      =  os.path.join(proj_dir, 'Intermediates', 'CIMMYT')
os.makedirs(out_dir, exist_ok=True)

# ── Inputs ────────────────────────────────────────────────────────────────────
logbook_f    =  os.path.join(cimmyt_dir, '1.-Farmer_Plot_Logbook_2012-2022_03.xlsx')   # plot locations & farmer info
yields_f     =  os.path.join(cimmyt_dir, '2.-Sowing_harvest_yields_2012-2022_02.xlsx')  # sowing, harvest, yield data

# ── Outputs ───────────────────────────────────────────────────────────────────
shp_out      =  os.path.join(out_dir, 'cimmyt_plot_locations.shp')
yields_out   =  os.path.join(out_dir, 'cimmyt_maize_yields.csv')

#### Load and clean the plot logbook

In [2]:
logbook = pd.read_excel(logbook_f, sheet_name='Data', header=1, engine='openpyxl')
print(f'Logbook rows: {len(logbook):,}  |  Unique plots: {logbook["PLOT ID"].nunique():,}')

Logbook rows: 69,008  |  Unique plots: 51,157


In [3]:
# Keep one row per plot with its GPS coordinates, state, and municipality.
# Coordinates are in LATITUDE N / LONGITUDE W (already signed negative for W).
loc_cols = ['PLOT ID', 'LATITUDE N', 'LONGITUDE W',
            'STATE', 'MUNICIPALITY',
            'ID_STATE (INEGI)', 'ID_MUNICIPALITY (INEGI)']

plots = (logbook[loc_cols]
         .dropna(subset=['LATITUDE N', 'LONGITUDE W'])
         .drop_duplicates(subset='PLOT ID')
         .reset_index(drop=True))

print(f'Unique plots with GPS coordinates: {len(plots):,}')

Unique plots with GPS coordinates: 51,000


#### Build and export the plot-locations shapefile

In [4]:
geometry = [Point(lon, lat) for lon, lat in zip(plots['LONGITUDE W'], plots['LATITUDE N'])]

# Rename columns to fit shapefile 10-char field-name limit
plots_shp = plots.rename(columns={
    'PLOT ID':                    'plot_id',
    'LATITUDE N':                 'lat',
    'LONGITUDE W':                'lon',
    'STATE':                      'state',
    'MUNICIPALITY':               'municipio',
    'ID_STATE (INEGI)':           'id_state',
    'ID_MUNICIPALITY (INEGI)':    'id_mun',
})

gdf = gpd.GeoDataFrame(plots_shp, geometry=geometry, crs='EPSG:4326')
gdf.to_file(shp_out)
print(f'Shapefile written: {shp_out}  ({len(gdf):,} features)')

Shapefile written: /home/j/Dropbox/Projects/Poppy_detection/Intermediates/CIMMYT/cimmyt_plot_locations.shp  (51,000 features)


#### Load and clean the sowing/harvest/yields data

In [5]:
yields = pd.read_excel(yields_f, sheet_name='Data', engine='openpyxl')
yields.columns = yields.columns.str.replace('\xa0', ' ')  # fix non-breaking spaces in header
print(f'Yields rows: {len(yields):,}')

Yields rows: 80,555


In [6]:
# Filter to maize grain yields reported in tons/ha
maize = yields[
    (yields['CROP'].str.upper() == 'MAIZE') &
    (yields['PRODUCT.OBTAINED'] == 'GRAIN') &
    (yields['UNIT.OF.MEASURE/HA'] == 'TON')
].copy()

print(f'Maize grain (ton/ha) rows: {len(maize):,}')

Maize grain (ton/ha) rows: 51,827


In [7]:
# Map season labels:
#   SUMMER → Spring-Summer  (Primavera-Verano, the main rainfed cycle)
#   WINTER → Fall-Winter    (Otono-Invierno, the irrigated cycle)
season_col =  'WINTER/SUMER. SEASON'
season_map =  {'SUMMER': 'Spring-Summer', 'WINTER': 'Fall-Winter'}

# Build clean output
out = maize[['PLOT.ID', 'YEAR', season_col, 'ACTUAL.YIELD.(UNIT/HA)']].copy()
out = out.rename(columns={
    'PLOT.ID':                'plot_id',
    'YEAR':                   'year',
    season_col:               'season_raw',
    'ACTUAL.YIELD.(UNIT/HA)': 'yield_ton_ha',
})
out['season'] = out['season_raw'].map(season_map)

# Drop rows with missing yield
out = out.dropna(subset=['yield_ton_ha']).reset_index(drop=True)

print(f'Clean maize yield obs: {len(out):,}')
print()
print(out['season'].value_counts())
print()
print(out['year'].value_counts().sort_index())
print()
print(out['yield_ton_ha'].describe())

Clean maize yield obs: 51,827

Spring-Summer    49781
Fall-Winter       2046
Name: season, dtype: int64

2012    9730
2013    5790
2014    2182
2015    2122
2016    7670
2017    9048
2018    9297
2019    3314
2020     924
2021    1168
2022     582
Name: year, dtype: int64

count    51827.000000
mean         4.835046
std          3.633291
min          0.000000
25%          2.000000
50%          4.000000
75%          6.700000
max         24.000000
Name: yield_ton_ha, dtype: float64


In [8]:
out.to_csv(yields_out, index=False)
print(f'Yields CSV written: {yields_out}  ({len(out):,} rows)')

Yields CSV written: /home/j/Dropbox/Projects/Poppy_detection/Intermediates/CIMMYT/cimmyt_maize_yields.csv  (51,827 rows)
